# Geosteering AI — Sprint A1.6: Benchmark JAX GPU Redesenhado

**Template:** `validate_jax_gpu_v240.ipynb` | **Sprint:** A1.6 (`A-jax-gpu-benchmark-redesign`)

Reescrita do notebook da Sprint A1 para usar a API `simulate_multi_jax_batched`
(Sprint A1.5, v2.42) e corrigir 8 bugs metodológicos identificados na auditoria
`v2.40.4_auditoria_resultados_sprint_a1`.

**Pré-requisito:** Runtime GPU T4 ou superior (Runtime → Change runtime type → GPU).

| Mudança vs v2.40 | Bug fixed | Onde |
|:--|:-:|:--|
| API batched — 1 chamada substitui loop Python | — | § 6 |
| Warmup por-cenário com shape exata + batch size de benchmark | C1 | § 3 |
| Run 1 (cold) reportado separado da mediana hot | C2 | § 6 |
| `jax.default_backend()` em vez de `d.platform == "gpu"` | H1 | § 2 |
| Leitura de `result.H_tensor.shape` força materialização | H2 | § 6 |
| `_UNIFIED_JIT_CACHE` invariante a modelos (`(n, npt)`) | H3 | § 3 (design) |
| Cenário H 87.5° vs Numba 90° documentado (validador JAX cap 89°) | M1 | § 5 (doc) |
| Batched cache não chaveia em `esp` (resolvido por design) | M2 | § 6 (design) |
| Baseline Numba medido localmente no T4 via subprocess CLI | M3 | § 5 |

**Critérios de aprovação (gate):**
- JAX batched **hot median** ≥ 1.5× Numba CPU **medido no MESMO T4** em A, B, E
- `pytest tests/test_simulation_jax_*.py -m gpu` → 100% PASS
- Sem OOM para N_MODELS=50, n_pos=600 (Cenário E)


In [ ]:
# === L5: Configurações JAX OBRIGATÓRIAS antes de qualquer import ===
# Equivalente ao 'NUMBA_NUM_THREADS no PAI' (L5 do mapeamento Numba→JAX):
# jax_enable_x64 e jax_compilation_cache_dir devem ser definidos ANTES
# de 'import jax' — após o import, jax_enable_x64 fica imutável.
import os

os.environ["JAX_ENABLE_X64"] = "1"                              # float64 — paridade Fortran <1e-12 requer float64
os.environ["JAX_PLATFORMS"] = "cuda,cpu"                        # MED-4: explicit em vez de "" (compat JAX 0.4.30+)
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/content/jax_cache"  # L8: cache XLA em SSD Colab (acelera Run 1 em re-execuções)

print("JAX env vars configurados (antes de qualquer import):")
print(f"  JAX_ENABLE_X64             = {os.environ['JAX_ENABLE_X64']}")
print(f"  JAX_PLATFORMS              = {os.environ['JAX_PLATFORMS']!r}  (CUDA preferido, CPU fallback)")
print(f"  JAX_COMPILATION_CACHE_DIR  = {os.environ['JAX_COMPILATION_CACHE_DIR']}")


## § 1 — Configurações da Sprint


In [ ]:
# Tag do repositório a clonar no Colab.
# Default "main" para desenvolvimento; mude para "v2.43" após a tag ser publicada.
GIT_TAG          = "main"
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/Geosteering_AI/sprint_a16/"  # destino dos resultados
PARIDADE_TOL     = 1e-12   # gate de paridade rigoroso (inviolável)
N_RUNS_HOT       = 4       # runs hot-cache (Runs 2-5 após Run 1 cold)
N_MODELS         = 50      # modelos geológicos por run (5 camadas, semente 42)
SEED             = 42      # reprodutibilidade — idêntico ao CLI _build_models

# Workers/threads pinados para o CLI Numba baseline (MED-5).
# T4 Colab é n1-standard-4 (4 vCPUs, sem HT). 4w × 1t evita oversubscription.
NUMBA_BASELINE_WORKERS = 4
NUMBA_BASELINE_THREADS = 1

# Diretório de cache JAX (L8): deve existir antes de import jax
os.makedirs("/content/jax_cache", exist_ok=True)

# --- Cenários canônicos (espelha geosteering_ai/cli/benchmark.py:113-147) ---
# dips_jax: versão segura para validador JAX (max 89.0°) — M1 doc
# numba_baseline_historical_mod_h: medição histórica Intel i9-9980HK Mac Intel (informativo, NÃO usado no gate)
SCENARIOS_JAX = {
    "A": {
        "n_pos": 1,
        "freqs": (20000.0,),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 1_180_000,
    },
    "B": {
        "n_pos": 100,
        "freqs": (20000.0,),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 320_000,
    },
    "C": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 119_000,
    },
    "D": {
        "n_pos": 1,
        "freqs": (20000.0,),
        "trs":   (0.5, 1.0, 1.5, 2.0),
        "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": None,
    },
    "E": {
        "n_pos": 600,
        "freqs": (20000.0,),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": 122_000,
    },
    "F": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs":   (0.5, 1.0, 1.5, 2.0),
        "dips_jax": (0.0,),
        "numba_baseline_historical_mod_h": None,
    },
    "G": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs":   (0.5, 1.0, 1.5, 2.0),
        "dips_jax": (0.0, 15.0, 30.0, 45.0),
        "numba_baseline_historical_mod_h": 45_000,
    },
    "H": {
        # M1: JAX validador cap dip em 89° (multi_forward.py:880).
        # Numba CLI usa 90° (benchmark.py:145). H NÃO entra no gate.
        "n_pos": 100,
        "freqs": (1e3, 2e3, 5e3, 1e4, 2e4, 5e4, 1e5, 2e5),
        "trs":   (0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5),
        "dips_jax": (0.0, 12.5, 25.0, 37.5, 50.0, 62.5, 75.0, 87.5),
        "numba_baseline_historical_mod_h": None,
    },
}

# Modelos por cenário calibrados para T4 (15 GB VRAM).
# Memória XLA ∝ n_models × nTR × nAng (eixos simultâneos via jax.vmap).
# F (4TR×1Ang): 50 modelos → OOM 17.8 GB; reduzido para 20.
# G (4TR×4Ang): 4× mais pesado que F; 5 modelos → ~7 GB.
# H (8TR×8Ang): stress test; 5 modelos → margem apertada, try/except no warmup.
N_MODELS_PER_SCENARIO = {
    "A": N_MODELS,   # n_pos=1,   combos=1   — gate
    "B": N_MODELS,   # n_pos=100, combos=1   — gate
    "C": N_MODELS,   # n_pos=100, combos=4
    "D": N_MODELS,   # n_pos=1,   combos=4
    "E": N_MODELS,   # n_pos=600, combos=1   — gate
    "F": 20,         # n_pos=100, combos=16  — OOM com 50 no T4 (nTR=4, nAng=1)
    "G": 5,          # n_pos=100, combos=64  — muito pesado (nTR=4, nAng=4)
    "H": 5,          # n_pos=100, combos=512 — stress test (nTR=8, nAng=8)
}

# Timeout do subprocess CLI por cenário (HIGH-2 fix).
# H é stress test (512 combos × 10 modelos em 4 vCPUs) — bump para 1800s.
NUMBA_CLI_TIMEOUT_S = {scen: (1800 if scen == "H" else 600) for scen in SCENARIOS_JAX}

print(f"GIT_TAG={GIT_TAG!r} | N_MODELS={N_MODELS} | N_RUNS_HOT={N_RUNS_HOT} (+1 cold) | tol={PARIDADE_TOL:.0e}")
print(f"Numba CLI baseline: workers={NUMBA_BASELINE_WORKERS} threads={NUMBA_BASELINE_THREADS}")
print(f"Cenários: {list(SCENARIOS_JAX.keys())}")
print(f"N_MODELS por cenário: {N_MODELS_PER_SCENARIO}")


## § 2 — Setup do Ambiente

Clona o repositório, instala dependências e valida que a GPU está
disponível com float64. **H1 fix**: detecção via `jax.default_backend()`
(compatível com JAX 0.4.14+) em vez do deprecado `device.platform == "gpu"`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Clonar repositório e instalar dependências
!git clone --depth 1 --branch {GIT_TAG} https://github.com/daniel-guitarplayer-8/geosteering-ai.git
%cd geosteering-ai
!pip install -q -e ".[all]"
!pip install -q pytest-json-report

# Imports — apenas APÓS as env vars da Célula 1
import json
import datetime
import re
import statistics
import subprocess
import time
from typing import Optional

import jax
import jax.numpy as jnp
import numpy as np

# Validar GPU — H1 fix: usar jax.default_backend() (API estável JAX 0.4.14+)
# em vez de d.platform == "gpu" (pode retornar "cuda" em builds recentes).
_BACKEND = jax.default_backend()
_DEVICES = jax.devices()
_GPU_DEVS = [d for d in _DEVICES if "gpu" in str(d).lower() or "cuda" in str(d).lower()]

assert _BACKEND in ("gpu", "cuda"), (
    f"GPU não é o backend default! backend={_BACKEND!r}, devices={_DEVICES}. "
    "Verificar Runtime → Change runtime type → GPU."
)
assert _GPU_DEVS, f"Nenhum device GPU/CUDA detectado. Devices: {_DEVICES}."

print(f"Backend default: {_BACKEND!r} ✓")
print(f"Dispositivos GPU: {_GPU_DEVS}")

# Validar float64
assert jax.config.jax_enable_x64, (
    "ERRO CRÍTICO: float64 não habilitado! "
    "JAX_ENABLE_X64 deve ser setado ANTES de 'import jax' (Célula 1)."
)
print(f"float64 habilitado: {jax.config.jax_enable_x64} ✓")

# Hardware info
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

# Pin commit hash (imutável — tag pode ser movida, hash não)
try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd="/content/geosteering-ai"
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    GIT_COMMIT = f"unknown-{GIT_TAG}"
print(f"Commit: {GIT_COMMIT[:12]}")


## § 3 — Warmup por-cenário com Batched API (C1 + H3 fixes)

**Por que warmup por-cenário com batch size exato é crítico:**

A v2.40 usava um warmup global com shape fixo `(nf=2, nTR=3, nAng=2)` que
**nenhum cenário utilizava** — todo Run 1 de cada cenário recompilava JIT
XLA (35-119× cold-start em B/C/E). Pior ainda, o warmup chamava
`simulate_multi_jax(models[0], ...)` em loop Python, e o `_BUCKET_JIT_CACHE`
(chaveado em `(camad_t, camad_r, n, npt)`) acumulava ~50 buckets distintos
durante Run 1 (1 por modelo aleatório).

**Solução A1.5 (v2.42) + A1.6 (warmup com batch correto):**
- `simulate_multi_jax_batched` usa `_UNIFIED_JIT_CACHE` keyed `(n, npt)` —
  invariante a valores de modelo (resolve bucket explosion / H3).
- **Mas o XLA cache** (subjacente ao `jax.jit` retornado) ainda chaveia
  no **shape completo dos args**, incluindo o `n_models` (axis size da
  outer `jax.vmap`). Warmup com batch=2 NÃO cobre benchmark com batch=50.
- **CRIT-2 fix**: warmup usa `np.tile(template, (N_MODELS_PER_SCENARIO[s], 1))`
  por cenário — batch exato do benchmark.

Lição L4 do simulador Numba aplicada: aquecer com dados anisotrópicos
(ρ_v ≠ ρ_h) + dip ≠ 0° para acionar todos os paths do código.


In [ ]:
from geosteering_ai.simulation import simulate_multi_jax_batched

# Modelo anisotrópico com dip ≠ 0° (cobre paths JIT que isotropico+dip=0 não acionam — L4)
_RHO_H_WARM_TEMPLATE = np.array([2.0, 15.0, 120.0, 8.0, 40.0], dtype=np.float64)
_RHO_V_WARM_TEMPLATE = _RHO_H_WARM_TEMPLATE * np.array([1.8, 2.5, 3.0, 1.5, 2.0], dtype=np.float64)
_ESP_WARM_TEMPLATE   = np.array([6.0, 9.0, 7.0], dtype=np.float64)   # 5 camadas → 3 espessuras internas

print("=" * 75)
print("WARMUP por-cenário com batch size EXATO (XLA cache key = shape completa)")
print("=" * 75)

for scen_name, scen in SCENARIOS_JAX.items():
    n_models_scen = N_MODELS_PER_SCENARIO[scen_name]
    n_pos = scen["n_pos"]
    n_combos = len(scen["freqs"]) * len(scen["trs"]) * len(scen["dips_jax"])

    # CRIT-2 fix: tile template para batch EXATO do benchmark
    rho_h_warm = np.tile(_RHO_H_WARM_TEMPLATE, (n_models_scen, 1))  # (n_models_scen, 5)
    rho_v_warm = np.tile(_RHO_V_WARM_TEMPLATE, (n_models_scen, 1))  # (n_models_scen, 5)
    esp_warm   = np.tile(_ESP_WARM_TEMPLATE,   (n_models_scen, 1))  # (n_models_scen, 3)

    pos = np.linspace(-5.0, 5.0, n_pos).astype(np.float64)  # match CLI benchmark.py:259

    t0 = time.perf_counter()
    try:
        res = simulate_multi_jax_batched(
            rho_h_warm, rho_v_warm, esp_warm, pos,
            frequencies_hz=list(scen["freqs"]),
            tr_spacings_m=list(scen["trs"]),
            dip_degs=list(scen["dips_jax"]),
        )
        _ = res.H_tensor.shape  # H2 defensivo
        print(f"  {scen_name}: n_models={n_models_scen:>2d}, n_pos={n_pos:>3d}, combos={n_combos:>3d} (nf={len(scen['freqs'])}, nTR={len(scen['trs'])}, nAng={len(scen['dips_jax'])}): {time.perf_counter()-t0:6.2f}s")
    except Exception as _exc:
        if "RESOURCE_EXHAUSTED" in str(_exc) or "Out of memory" in str(_exc):
            print(f"  {scen_name}: OOM com n_models={n_models_scen} — warmup ignorado; Run 1 será cold-start")
        else:
            raise

print("\n✓ Warmup concluído. XLA cache populado para batch exato de cada cenário.")
print("   Run 1 do benchmark deve estar HOT — se cold/hot ratio > 2×, warmup falhou.")


## § 4 — Paridade Fortran < 1e-12 (testes pytest -m gpu)

Os testes `@pytest.mark.gpu` validam paridade JAX GPU vs Fortran < 1e-12.
Com `JAX_PLATFORMS="cuda,cpu"` (Célula 1), o conftest detecta GPU
e executa os testes que seriam SKIPPED em CPU.


In [ ]:
# Executa testes JAX GPU com relatório JSON
!python -m pytest tests/test_simulation_jax_*.py \
    -m gpu \
    -v \
    --tb=short \
    --json-report \
    --json-report-file=/tmp/jax_gpu_pytest.json \
    2>&1 | tail -40

# Carregar relatório pytest UMA VEZ — reutilizado em § 8 e § 9 (MED-3 fix)
with open("/tmp/jax_gpu_pytest.json") as _f:
    _pytest_report = json.load(_f)
_summary = _pytest_report["summary"]
print(f"\nResumo pytest: {_summary}")
assert _summary.get("failed", 0) == 0, (
    f"GATE PARIDADE FALHOU: {_summary['failed']} testes falharam. "
    "Investigar antes de prosseguir com benchmark."
)
print(f"✓ Paridade: {_summary.get('passed', 0)} PASS / 0 FAIL")


In [ ]:
# Sanity check da API batched (shape, dtype, ausência de NaN/Inf)
_n_models_check = 3
_rho_h_check = np.tile(np.array([1.0, 10.0, 100.0, 10.0, 1.0]), (_n_models_check, 1))
_rho_v_check = _rho_h_check * 2.0  # TIV anisotrópico
_esp_check   = np.tile(np.array([5.0, 10.0, 5.0]), (_n_models_check, 1))
_pos_check   = np.linspace(-5.0, 5.0, 50).astype(np.float64)

_res = simulate_multi_jax_batched(
    _rho_h_check, _rho_v_check, _esp_check, _pos_check,
    frequencies_hz=[20000.0], tr_spacings_m=[1.0], dip_degs=[0.0],
)
_ = _res.H_tensor.shape  # H2 defensivo

assert _res.H_tensor.dtype == np.complex128,                    f"dtype errado: {_res.H_tensor.dtype}"
assert _res.H_tensor.shape == (_n_models_check, 1, 1, 50, 1, 9), f"shape errado: {_res.H_tensor.shape}"
assert not np.any(np.isnan(_res.H_tensor.real)),                 "NaN em H_tensor.real!"
assert not np.any(np.isnan(_res.H_tensor.imag)),                 "NaN em H_tensor.imag!"
assert not np.any(np.isinf(_res.H_tensor)),                      "Inf em H_tensor!"

print(f"Shape:  {_res.H_tensor.shape}  ✓  (n_models={_n_models_check})")
print(f"dtype:  {_res.H_tensor.dtype}  ✓")
print(f"NaN:    ausente  ✓")
print(f"Inf:    ausente  ✓")
print(f"|Hzz| médio (modelo 0): {np.abs(_res.H_tensor[0,0,0,:,0,8]).mean():.6e}  (componente axial ZZ)")


## § 5 — Baseline Numba CPU no MESMO hardware T4 (M3 fix)

A auditoria v2.40.4 identificou que o gate ≥1.5× Numba era inválido
porque o baseline estava medido em hardware diferente (Intel i9-9980HK
8C/16T Mac Intel) — Colab T4 usa `n1-standard-4` (4 vCPUs Xeon, sem HT),
~50-60% do throughput Numba do Mac Intel.

**M3 fix**: mede o baseline Numba **localmente no T4** via subprocess
`geosteering-cli benchmark --scenario X --n N --workers 4 --threads 1`.
Workers/threads pinados (MED-5 fix) para reprodutibilidade — sem isto, o
auto-detect `recommend_default_parallelism` heuristic pode variar entre
sessões Colab. O CLI usa `simulate_multi` (Numba ProcessPool) idêntico
ao código de produção. Output formato `"Cenário X — N,NNN,NNN mod/h"` é
parseado via regex robusto.

**Cenário H — nota M1**: o validador JAX limita `dip_degs ≤ 89°`
(ver `multi_forward.py:880`), enquanto o CLI Numba usa 90°. Os
throughputs **não são exatamente** apples-to-apples em H — diferença
documentada e gate só avaliado em A/B/E.

**Cenários D/F/H**: baselines podem retornar `None` se o CLI subprocess
falhar (timeout/erro). Esses cenários ficam fora do gate por padrão.
Cenário H tem timeout estendido (1800s — HIGH-2 fix).


In [ ]:
def measure_numba_baseline_t4(
    scen_name: str,
    n_models: int,
    timeout_s: int,
    workers: int = NUMBA_BASELINE_WORKERS,
    threads: int = NUMBA_BASELINE_THREADS,
) -> Optional[float]:
    """Mede baseline Numba CPU no MESMO hardware T4 via CLI subprocess (M3 fix).

    Invoca ``geosteering-cli benchmark --scenario X --n N --workers W --threads T``
    e parseia o throughput em mod/h. Usa os cenários canônicos do CLI (A-H)
    — não passa --frequencies/--dips/--tr-spacings (mantém defaults).

    Args:
        scen_name: identificador do cenário (A..H, deve existir no CLI)
        n_models: número de modelos
        timeout_s: timeout do subprocess em segundos
        workers: --workers (default 4, pinado MED-5)
        threads: --threads (default 1, pinado MED-5)

    Returns:
        Throughput em mod/h (float) ou None em caso de erro/timeout/CLI ausente.
    """
    cmd = [
        "geosteering-cli", "benchmark",
        "--scenario", scen_name,
        "--n", str(n_models),
        "--workers", str(workers),
        "--threads", str(threads),
    ]
    try:
        result = subprocess.run(
            cmd, capture_output=True, text=True, check=True, timeout=timeout_s,
        )
    except subprocess.CalledProcessError as exc:
        print(f"    [WARN] CLI exit {exc.returncode}: {(exc.stderr or '')[:200]}")
        return None
    except subprocess.TimeoutExpired:
        print(f"    [WARN] CLI timeout (>{timeout_s}s)")
        return None
    except (FileNotFoundError, OSError) as exc:
        # HIGH-1 fix: geosteering-cli não em PATH ou erro de OS
        print(f"    [WARN] CLI não encontrado/erro OS: {exc}")
        return None

    # Parse "Cenário X — 1,180,000 mod/h" (benchmark.py:319)
    match = re.search(r"—\s+([\d,]+)\s+mod/h", result.stdout)
    if not match:
        print(f"    [WARN] Parse falhou em stdout: {result.stdout[:200]!r}")
        return None
    return float(match.group(1).replace(",", ""))


print("=" * 75)
print(f"MEDINDO BASELINE NUMBA CPU NO T4 LOCAL  (workers={NUMBA_BASELINE_WORKERS}, threads={NUMBA_BASELINE_THREADS})")
print("=" * 75)

numba_t4_baselines = {}
for scen_name in SCENARIOS_JAX:
    n_m = N_MODELS_PER_SCENARIO[scen_name]
    timeout = NUMBA_CLI_TIMEOUT_S[scen_name]
    print(f"\n  Cenário {scen_name} (n={n_m}, timeout={timeout}s)...", end=" ", flush=True)
    t0 = time.perf_counter()
    th = measure_numba_baseline_t4(scen_name, n_m, timeout_s=timeout)
    elapsed = time.perf_counter() - t0
    numba_t4_baselines[scen_name] = th
    if th is not None:
        print(f"{th:>12,.0f} mod/h  ({elapsed:.1f}s)")
    else:
        print(f"FALHA ({elapsed:.1f}s) — gate cenário {scen_name} indisponível")

print("\n✓ Baseline Numba T4 LOCAL medido.")


## § 6 — Benchmark JAX batched (C1 + C2 + H2 + M2 fixes)

Mede throughput em mod/h da API `simulate_multi_jax_batched` (Sprint A1.5,
v2.42) que aplica `jax.vmap` sobre o eixo `n_models` — substitui o loop
Python `for m in models: simulate_multi_jax(...)` por **1 trace XLA + 1
sync GPU→CPU** para todo o batch.

**C2 fix**: Run 1 (cold-start, pode triggerar compile XLA se warmup não
cobriu) é reportado SEPARADO de Runs 2-5 (hot-cache). O gate usa
`statistics.median(runs_2_to_5)`, NÃO `statistics.median([cold] + hot)`.

**H2 fix**: leitura de `result.H_tensor.shape` após cada call força
materialização (defensivo — a função já chama `block_until_ready()` +
`np.asarray()` internamente em `multi_forward.py:1111-1112`).

**M2 fix (design)**: `_UNIFIED_JIT_CACHE` chaveado `(n, npt)` é invariante
a `esp_batch` — variância de espessuras NÃO causa recompile. RNG mantido
para reprodutibilidade com CLI.

**Métricas reportadas:**
- `ratio_cold_to_hot`: cold/hot. ≈ 1.0 indica warmup efetivo; > 2× indica
  cold-start residual (warmup falhou ou XLA cache evicted).
- `latency_ms_per_model_hot`: `3.6e6 / median_hot` ms — tempo por modelo
  (CRIT-1 fix: fórmula correta sem multiplicar por n_models).


In [ ]:
def _build_models(n_models: int, seed: int = 42) -> list:
    """Gera modelos 5-camada determinísticos (idêntico ao CLI _build_models).

    Mantém paridade EXATA com geosteering_ai/cli/benchmark.py:150-184
    (sentinel: rng.uniform(1.0, 100.0, size=5) para rho_h, * uniform(1, 3)
    para rho_v, uniform(2.0, 10.0, size=3) para esp). Qualquer mudança no
    CLI _build_models DEVE ser espelhada aqui para preservar a comparação
    apples-to-apples Numba CLI vs JAX batched.
    """
    rng = np.random.default_rng(seed)
    models = []
    for _ in range(n_models):
        rho_h = rng.uniform(1.0, 100.0, size=5).astype(np.float64)
        rho_v = rho_h * rng.uniform(1.0, 3.0, size=5).astype(np.float64)
        esp   = rng.uniform(2.0, 10.0, size=3).astype(np.float64)  # n=5 → 3 espessuras
        models.append({"rho_h": rho_h, "rho_v": rho_v, "esp": esp})
    return models


def run_benchmark_batched_scenario(scen_name: str, n_models: int, n_runs_hot: int = 4) -> dict:
    """Executa benchmark batched de um cenário com Run 1 cold isolado.

    Run 1 (cold): primeira chamada após warmup global; deve reusar XLA cache
    se warmup cobriu o batch size exato. Reportado separadamente.

    Runs 2-(n_runs_hot+1): hot-cache puro. statistics.median sobre estes
    é a métrica oficial do gate (C2 fix).

    Args:
        scen_name: identificador do cenário (A..H)
        n_models: número de modelos no batch
        n_runs_hot: runs hot-cache após Run 1 cold (default 4 → total 5)

    Returns:
        dict com run_1_cold_mod_h, hot_throughputs_mod_h, median_hot_mod_h,
        latency_ms_per_model_hot (CRIT-1 fix) e metadados.
    """
    scen = SCENARIOS_JAX[scen_name]
    n_pos = scen["n_pos"]
    positions_z = np.linspace(-5.0, 5.0, n_pos).astype(np.float64)  # match CLI benchmark.py:259
    models = _build_models(n_models, seed=SEED)

    rho_h_batch = np.stack([m["rho_h"] for m in models])  # (n_models, 5)
    rho_v_batch = np.stack([m["rho_v"] for m in models])  # (n_models, 5)
    esp_batch   = np.stack([m["esp"]   for m in models])  # (n_models, 3)

    kwargs = dict(
        frequencies_hz=list(scen["freqs"]),
        tr_spacings_m=list(scen["trs"]),
        dip_degs=list(scen["dips_jax"]),
    )

    # === Run 1 — COLD ===
    t0 = time.perf_counter()
    res = simulate_multi_jax_batched(rho_h_batch, rho_v_batch, esp_batch,
                                      positions_z, **kwargs)
    _ = res.H_tensor.shape  # H2 defensivo
    run_1_elapsed = time.perf_counter() - t0
    run_1_mod_h = (n_models / run_1_elapsed) * 3600.0
    print(f"    Run 1 (cold): {run_1_mod_h:>12,.0f} mod/h  ({run_1_elapsed:.3f}s)")

    # === Runs 2 a (n_runs_hot + 1) — HOT ===
    hot_throughputs = []
    for i in range(n_runs_hot):
        t0 = time.perf_counter()
        res = simulate_multi_jax_batched(rho_h_batch, rho_v_batch, esp_batch,
                                          positions_z, **kwargs)
        _ = res.H_tensor.shape  # H2 defensivo
        elapsed = time.perf_counter() - t0
        th = (n_models / elapsed) * 3600.0
        hot_throughputs.append(th)
        print(f"    Run {i+2}  (hot):  {th:>12,.0f} mod/h  ({elapsed:.3f}s)")

    median_hot = statistics.median(hot_throughputs)
    # CRIT-1 fix: ms POR MODELO = 3.6e6 ms/h / throughput em mod/h
    # (NÃO n_models / throughput * 3.6e6, que dá ms para TODOS os modelos)
    latency_ms_per_model_hot = 3_600_000.0 / median_hot

    return {
        "scenario": scen_name,
        "n_models": n_models,
        "n_pos": n_pos,
        "nf": len(scen["freqs"]),
        "nTR": len(scen["trs"]),
        "nAng": len(scen["dips_jax"]),
        "n_runs_hot": n_runs_hot,
        "run_1_cold_mod_h": run_1_mod_h,
        "run_1_cold_elapsed_s": run_1_elapsed,
        "hot_throughputs_mod_h": hot_throughputs,
        "median_hot_mod_h": median_hot,
        "min_hot_mod_h": min(hot_throughputs),
        "max_hot_mod_h": max(hot_throughputs),
        "ratio_cold_to_hot": run_1_mod_h / median_hot,
        "latency_ms_per_model_hot": latency_ms_per_model_hot,
        "numba_t4_local_mod_h": numba_t4_baselines.get(scen_name),
        "numba_baseline_historical_mod_h": scen["numba_baseline_historical_mod_h"],
    }


results_batched = {}
print("=" * 75)
print("BENCHMARK — simulate_multi_jax_batched (Sprint A1.5, v2.42)")
print("=" * 75)

for scen_name in SCENARIOS_JAX:
    n_m = N_MODELS_PER_SCENARIO[scen_name]
    scen = SCENARIOS_JAX[scen_name]
    n_combos = len(scen["trs"]) * len(scen["dips_jax"]) * len(scen["freqs"])
    print(f"\nCenário {scen_name} — n_pos={scen['n_pos']}, {n_combos} combos (nf×nTR×nAng), n_models={n_m}")
    try:
        results_batched[scen_name] = run_benchmark_batched_scenario(
            scen_name, n_models=n_m, n_runs_hot=N_RUNS_HOT,
        )
    except Exception as _exc:
        if "RESOURCE_EXHAUSTED" in str(_exc) or "Out of memory" in str(_exc):
            print(f"  → OOM com n_models={n_m} — cenário {scen_name} pulado")
            results_batched[scen_name] = {
                "scenario": scen_name, "n_models": n_m, "n_pos": scen["n_pos"],
                "nf": len(scen["freqs"]), "nTR": len(scen["trs"]), "nAng": len(scen["dips_jax"]),
                "n_runs_hot": N_RUNS_HOT, "run_1_cold_mod_h": None, "run_1_cold_elapsed_s": None,
                "hot_throughputs_mod_h": [], "median_hot_mod_h": None,
                "min_hot_mod_h": None, "max_hot_mod_h": None, "ratio_cold_to_hot": None,
                "latency_ms_per_model_hot": None, "numba_t4_local_mod_h": None,
                "oom": True,
            }
            continue
        else:
            raise
    r = results_batched[scen_name]
    nb_t4_local = r["numba_t4_local_mod_h"]
    if scen_name == "H":
        # M1: H comparison não é apples-to-apples (87.5° JAX vs 90° Numba)
        ratio_str = "  (Numba T4: 87.5° vs 90° — M1 doc)" if nb_t4_local else "  (Numba T4 indisponível)"
    else:
        ratio_str = f"  ({r['median_hot_mod_h']/nb_t4_local:.2f}× Numba T4)" if nb_t4_local else "  (Numba T4 indisponível)"
    print(f"  → Hot median: {r['median_hot_mod_h']:>12,.0f} mod/h{ratio_str}")
    print(f"    Cold/hot ratio: {r['ratio_cold_to_hot']:.2f}  (≈1.0 = warmup efetivo, >2× = warmup falhou)")
    print(f"    Latência: {r['latency_ms_per_model_hot']:.3f} ms/modelo")

print("\n✓ Benchmark batched concluído.")


## § 7 — Tabela comparativa + Gate ≥1.5× Numba T4 LOCAL

Gate avaliado em A, B, E (cenários canônicos single-TR, single-dip,
freq 20kHz). Cenários D, F, H podem retornar baseline `None` se o CLI
subprocess falhou — esses ficam fora do gate. **Cenário H NÃO entra no
gate** (dip 87.5° JAX vs 90° Numba — M1 doc); razão impressa entre
parênteses como informativa.

| Métrica | Fonte | Uso |
|:--|:--|:--|
| `numba_t4_local_mod_h` | Subprocess CLI medido no T4 (M3 fix) | **Gate** |
| `median_hot_mod_h` | `statistics.median([Run 2..5])` (C2 fix) | **Gate** |
| `run_1_cold_mod_h` | Run 1 isolado (cold-start) | Informativo |
| `ratio_cold_to_hot` | cold / hot | Diagnóstico warmup (esperado ≈1.0) |
| `numba_baseline_historical_mod_h` | Intel i9-9980HK Mac Intel (referência) | Apenas referência |


In [ ]:
print("=" * 120)
print(f"{'Cen':>3} | {'n_pos':>5} | {'nf×TR×Ang':>9} | {'Numba T4 LOCAL':>14} | {'JAX cold (Run1)':>15} | {'JAX hot (med)':>14} | {'Razão hot/T4':>12} | {'C/H':>5} | {'Hist. i9':>10}")
print("-" * 120)

GATE_PASS_SCENARIOS = ["A", "B", "E"]  # cenários canônicos com critério ≥1.5×
gate_results = {}

for scen_name in SCENARIOS_JAX:
    scen   = SCENARIOS_JAX[scen_name]
    r      = results_batched[scen_name]
    nb_t4  = r["numba_t4_local_mod_h"]
    nb_hist = scen["numba_baseline_historical_mod_h"]
    combos = f"{len(scen['freqs'])}×{len(scen['trs'])}×{len(scen['dips_jax'])}"

    cold_str = f"{r['run_1_cold_mod_h']:>13,.0f}"
    hot_str  = f"{r['median_hot_mod_h']:>12,.0f}"
    nb_str   = f"{nb_t4:>12,.0f}" if nb_t4 else f"{'(falha)':>12}"
    # MED-1: H tem dip mismatch — não imprime ratio direto
    if scen_name == "H" and nb_t4:
        ratio_str = f"{'(dip≠)':>11}"  # 87.5° JAX vs 90° Numba
    else:
        ratio_str = f"{r['median_hot_mod_h']/nb_t4:>10.2f}×" if nb_t4 else f"{'—':>11}"
    ch_str = f"{r['ratio_cold_to_hot']:>5.2f}"
    hist_str = f"{nb_hist:>9,.0f}" if nb_hist else f"{'—':>10}"

    print(f"{scen_name:>3} | {scen['n_pos']:>5} | {combos:>9} | {nb_str:>14} m/h | {cold_str} m/h | {hot_str} m/h | {ratio_str} | {ch_str} | {hist_str}")

    if scen_name in GATE_PASS_SCENARIOS and nb_t4:
        ratio = r["median_hot_mod_h"] / nb_t4
        gate_results[scen_name] = {
            "median_hot_mod_h": r["median_hot_mod_h"],
            "numba_t4_local_mod_h": nb_t4,
            "ratio_hot_vs_t4": ratio,
            "pass": ratio >= 1.5,
        }

print("=" * 120)
print("  C/H = ratio cold/hot (Run1 / median hot). ≈1.0 = warmup efetivo. >2× = warmup falhou.")
print("  H (cenário stress): JAX usa dip 87.5° (cap 89° validador), Numba usa 90° — razão não comparável (M1 doc).")

print("\n=== Gate de Aprovação (≥ 1.5× Numba T4 LOCAL em A, B, E) ===")
all_pass = True
gate_evaluable = False
for scen_name in GATE_PASS_SCENARIOS:
    if scen_name not in gate_results:
        print(f"  Cenário {scen_name}: Numba T4 LOCAL indisponível → gate não avaliável")
        continue
    g = gate_results[scen_name]
    gate_evaluable = True
    status = "✓ PASS" if g["pass"] else "✗ FAIL"
    print(f"  Cenário {scen_name}: {g['median_hot_mod_h']:>12,.0f} mod/h ({g['ratio_hot_vs_t4']:.2f}× Numba T4) → {status}")
    if not g["pass"]:
        all_pass = False

if not gate_evaluable:
    print("\n? GATE NÃO AVALIÁVEL: nenhum baseline Numba T4 disponível em A/B/E.")
elif all_pass:
    print("\n✓ GATE APROVADO: JAX batched supera 1.5× Numba T4 nos cenários A, B, E.")
else:
    print("\n✗ GATE REPROVADO: investigar antes de prosseguir para Sprint A2.")


## § 8 — Exportar Resultados para Drive


In [ ]:
# Detectar hardware (T4 / A100 / etc.)
try:
    _gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
    ).decode().strip().lower()
    RUNTIME_LABEL = "a100" if "a100" in _gpu_name else ("t4" if "t4" in _gpu_name else _gpu_name.replace(" ", "_"))
except Exception:
    RUNTIME_LABEL = "gpu_unknown"
print(f"Hardware detectado: {RUNTIME_LABEL}")

ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
out_fname = f"sprint_a16_jax_batched_benchmark_{RUNTIME_LABEL}_{ts}.json"
out_path  = os.path.join(OUTPUT_DRIVE_DIR, out_fname)

# MED-3 fix: reusa _pytest_report da Célula 9 (sem reload)
output = {
    "sprint": "A1.6 — A-jax-gpu-benchmark-redesign",
    "template": "validate_jax_gpu_v240 (Sprint A1.6 rewrite)",
    "git_tag": GIT_TAG,
    "git_commit": GIT_COMMIT,
    "runtime": RUNTIME_LABEL,
    "timestamp_utc": ts,
    "config": {
        "n_models_per_scenario": N_MODELS_PER_SCENARIO,
        "n_runs_hot": N_RUNS_HOT,
        "seed": SEED,
        "paridade_tol": PARIDADE_TOL,
        "jax_enable_x64": True,
        "api": "simulate_multi_jax_batched (Sprint A1.5 v2.42)",
        "positions_z_range": "[-5.0, 5.0] (match CLI benchmark.py:259)",
        "numba_baseline_workers": NUMBA_BASELINE_WORKERS,
        "numba_baseline_threads": NUMBA_BASELINE_THREADS,
        "numba_cli_timeout_s": NUMBA_CLI_TIMEOUT_S,
    },
    "pytest_summary": _pytest_report["summary"],
    "pytest_duration_sec": _pytest_report.get("duration", 0.0),
    "benchmark_batched": results_batched,
    "numba_t4_local_baselines": numba_t4_baselines,
    "gate_results": gate_results,
    "gate_pass": all_pass if gate_evaluable else None,
    "gate_evaluable": gate_evaluable,
    "h_scenario_note": "dip_degs max=87.5° (JAX validador cap 89°) vs 90° Numba CLI (M1 doc, gate só A/B/E)",
    "fixes_applied": {
        "C1": "warmup por-cenário com batch size EXATO (XLA cache key inclui n_models)",
        "C2": "Run 1 cold isolado de statistics.median(Runs 2-5)",
        "H1": "jax.default_backend() em vez de d.platform == 'gpu'",
        "H2": "result.H_tensor.shape após cada call (force materialization)",
        "H3": "_UNIFIED_JIT_CACHE keyed (n, npt) — invariante a modelos (design)",
        "M1": "doc: cap dip 89° JAX (multi_forward.py:880); H fora do gate; tabela anotada",
        "M2": "batched cache não chaveia em esp (design)",
        "M3": "Numba baseline medido localmente no T4 via geosteering-cli benchmark",
    },
    "review_fixes_applied": {
        "CRIT-1": "latency_ms_per_model_hot = 3.6e6 / median_hot (não n_models / hot * 3.6e6)",
        "CRIT-2": "warmup com batch size matching N_MODELS_PER_SCENARIO (XLA cache hit)",
        "HIGH-1": "FileNotFoundError/OSError capturados em measure_numba_baseline_t4",
        "HIGH-2": "H scenario timeout 1800s (stress test 8x8x8)",
        "MED-1":  "H row na tabela sem ratio direto (dip mismatch)",
        "MED-3":  "single _pytest_report load (sem _pytest_data duplicado)",
        "MED-4":  "JAX_PLATFORMS='cuda,cpu' explícito (compat JAX 0.4.30+)",
        "MED-5":  "--workers/--threads pinados no CLI Numba baseline",
    },
}

with open(out_path, "w") as _f:
    json.dump(output, _f, indent=2)

print(f"✓ Resultados exportados: {out_path}")
print(f"  Tamanho: {os.path.getsize(out_path) / 1024:.1f} KB")


## § 9 — Resumo Final


In [ ]:
_pytest_sum = _pytest_report["summary"]

print("=" * 80)
print("SPRINT A1.6 — BENCHMARK JAX GPU BATCHED (REDESIGN) CONCLUÍDA")
print(f"Hardware: {RUNTIME_LABEL.upper()} | Commit: {GIT_COMMIT[:12]}")
print("=" * 80)
print(f"\nParidade Fortran <1e-12 (pytest):")
print(f"  PASS:    {_pytest_sum.get('passed', 0)}")
print(f"  FAIL:    {_pytest_sum.get('failed', 0)}")
print(f"  SKIPPED: {_pytest_sum.get('skipped', 0)}")

print("\nThroughput (mediana hot) JAX batched vs Numba T4 LOCAL:")
for scen_name in SCENARIOS_JAX:
    r   = results_batched[scen_name]
    nb  = r["numba_t4_local_mod_h"]
    cold = r["run_1_cold_mod_h"]
    hot  = r["median_hot_mod_h"]
    if scen_name == "H":
        ratio_str = "  (H: dip≠ — M1)" if nb else "  (T4 indisponível)"
    else:
        ratio_str = f"  ({hot/nb:.2f}× Numba T4)" if nb else "  (T4 indisponível)"
    print(f"  Cenário {scen_name}: cold={cold:>10,.0f}  hot={hot:>12,.0f} mod/h{ratio_str}  C/H={r['ratio_cold_to_hot']:.2f}")

if gate_evaluable:
    print(f"\nGate ≥1.5× Numba T4 LOCAL (A, B, E): {'✓ APROVADO' if all_pass else '✗ REPROVADO'}")
else:
    print("\nGate: NÃO AVALIÁVEL (nenhum baseline Numba T4 LOCAL em A/B/E)")
print(f"\nResultados salvos em: {out_path}")
print("=" * 80)

print("""
PRÓXIMOS PASSOS:
  1. Reportar tabela de resultados para nova sessão Claude
  2. Claude preenche docs/PERFORMANCE_BASELINE.md seção JAX GPU
  3. Claude fecha Sprint A1.6 → cria docs/sprints/v2.43.md
  4. Claude atualiza ROADMAP §0: A-jax-gpu-benchmark-redesign → DONE
  5. Claude inicia Sprint A2: implement simulation.simulate(backend=...)
""")
